<a href="https://colab.research.google.com/github/beatrizdfs/portfolio_projetos/blob/main/DW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [453]:
import pandas as pd
import sqlite3

In [454]:
df = pd.read_excel("/content/base_vendas_xperiun.xlsx")

In [455]:
df.columns

Index(['Produto', 'Preço', 'Pgto.', 'Nº Parcelas', 'Data', 'Cliente', 'E-mail',
       'DDD', 'Telefone'],
      dtype='object')

In [456]:
df = df.rename(columns={
    "Produto" : "produto",
    "Preço" : "preco",
    "Pgto." : "forma_pagamento",
    "Nº Parcelas" : "num_parcelas",
    "Data" : "data",
    "Cliente" : "cliente",
    "E-mail" : "email",
    "DDD" : "ddd",
    "Telefone" : "telefone"
})
df

,produto,preco,forma_pagamento,num_parcelas,data,cliente,email,ddd,telefone
0,Produto A,1000,Cartão de Credito,1,2018-01-01,Jonatas Martins,jonatas@gmail.com,92,984300000
1,Produto A,500,Cartão de Credito,2,2018-01-01,Jhonns Martins,jhonns@hotmail.com,85,999000000
2,Produto B,1000,Cartão de Credito,12,2018-01-01,Samuel Martins,samuel@gmail.com,32,988500000
3,Produto A,500,Cartão de Credito,2,2018-01-01,Danubia Martins,danubia@gmail.com,31,989700000
4,Produto B,1000,Cartão de Credito,12,2018-01-01,Presley Martins,presley@hotmail.com,28,999500000
...,...,...,...,...,...,...,...,...,...
7004,Produto A,500,Cartão de Credito,4,2020-12-31,Vilson Martins,vilson@gmail.com,51,993700000
7005,Produto C,2000,Boleto Bancário,1,2020-12-31,Wanderico Martins,wanderico@hotmail.com,21,997900000
7006,Produto C,2000,Cartão de Credito,1,2020-12-31,Manoela Martins,manoela@gmail.com,41,998900000
7007,Produto A,500,Boleto Bancário,1,2020-12-31,Myrna Martins,myrna@hotmail.com,98,982900000


In [457]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7009 entries, 0 to 7008
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   produto          7009 non-null   object        
 1   preco            7009 non-null   int64         
 2   forma_pagamento  7009 non-null   object        
 3   num_parcelas     7009 non-null   int64         
 4   data             7009 non-null   datetime64[ns]
 5   cliente          7009 non-null   object        
 6   email            7009 non-null   object        
 7   ddd              7009 non-null   int64         
 8   telefone         7009 non-null   int64         
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 492.9+ KB


In [458]:
dim_cliente = (
    df[["cliente", "email", "ddd", "telefone"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [459]:
len(dim_cliente)

6027

In [460]:
dim_cliente.insert(0, "id_cliente", range(1, len(dim_cliente) + 1)) # o + 1 é usado pq o range ignora o último registro

In [461]:
dim_produto = (
    df[["produto"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [462]:
dim_produto.insert(0, "id_produto", range(1, len(dim_produto) + 1))

In [463]:
dim_pagamento = (
    df[["forma_pagamento", "num_parcelas"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [464]:
dim_pagamento.insert(0, "id_pagamento", range(1, len(dim_pagamento) + 1))

In [465]:
dim_calendario = (
    df[["data"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_calendario["data"] = pd.to_datetime(dim_calendario["data"]) #transforma em tipo date
dim_calendario["dia"] = dim_calendario["data"].dt.day
dim_calendario["mes"] = dim_calendario["data"].dt.month
dim_calendario["ano"] = dim_calendario["data"].dt.year

dim_calendario.insert(0, "id_data", range(1, len(dim_calendario) + 1))

In [466]:
dim_calendario

,id_data,data,dia,mes,ano
0,1,2018-01-01,1,1,2018
1,2,2018-01-02,2,1,2018
2,3,2018-01-03,3,1,2018
3,4,2018-01-04,4,1,2018
4,5,2018-01-05,5,1,2018
...,...,...,...,...,...
1090,1091,2020-12-27,27,12,2020
1091,1092,2020-12-28,28,12,2020
1092,1093,2020-12-29,29,12,2020
1093,1094,2020-12-30,30,12,2020


In [467]:
fato = (
    df
    .merge(dim_cliente, on= ["cliente", "email", "ddd", "telefone"], how= "left")
    .merge(dim_produto, on= ["produto"], how= "left")
    .merge(dim_pagamento, on= ["forma_pagamento", "num_parcelas"], how= "left")
    .merge(dim_calendario [["id_data", "data"]], on = ["data"], how= "left")
)

In [468]:
fato_vendas = fato[["id_cliente", "id_produto", "id_pagamento", "id_data", "preco"]]

In [469]:
fato_vendas.insert(0, "id_venda", range(1, len(fato_vendas) + 1))

In [470]:
db_name = "dw_vendas.db"
conn = sqlite3.connect(db_name)
cur = conn.cursor()

In [471]:
cur.execute("PRAGMA foreign_keys = ON;")

In [472]:
schema_sql = """
DROP TABLE IF EXISTS fato_vendas;
DROP TABLE IF EXISTS dim_cliente;
DROP TABLE IF EXISTS dim_produto;
DROP TABLE IF EXISTS dim_pagamento;
DROP TABLE IF EXISTS dim_calendario;

CREATE TABLE dim_cliente (
    id_cliente INTEGER PRIMARY KEY,
    cliente TEXT NOT NULL,
    email TEXT,
    ddd TEXT,
    telefone TEXT
);

CREATE TABLE dim_produto (
    id_produto INTEGER PRIMARY KEY,
    produto TEXT NOT NULL
);

CREATE TABLE dim_pagamento (
    id_pagamento INTEGER PRIMARY KEY,
    forma_pagamento TEXT,
    num_parcelas INTEGER
);

CREATE TABLE dim_calendario (
    id_data INTEGER PRIMARY KEY,
    data DATE,
    dia INTEGER,
    mes INTEGER,
    ano INTEGER
);

CREATE TABLE fato_vendas(
    id_venda INTEGER PRIMARY KEY,
    id_cliente INTEGER NOT NULL,
    id_produto INTEGER NOT NULL,
    id_pagamento INTEGER NOT NULL,
    id_data INTEGER NOT NULL,
    preco REAL NOT NULL,
    FOREIGN KEY (id_cliente) REFERENCES dim_cliente(id_cliente),
    FOREIGN KEY (id_produto) REFERENCES dim_produto(id_produto),
    FOREIGN KEY (id_pagamento) REFERENCES dim_pagamento(id_pagamento),
    FOREIGN KEY (id_data) REFERENCES dim_calendario(id_data)
    );
"""

In [473]:
cur.executescript(schema_sql)
conn.commit()

In [474]:
dim_cliente.to_sql("dim_cliente", conn, if_exists = "append", index = False)
dim_produto.to_sql("dim_produto", conn, if_exists = "append", index = False)
dim_pagamento.to_sql("dim_pagamento", conn, if_exists = "append", index = False)
dim_calendario.to_sql("dim_calendario", conn, if_exists = "append", index = False)
fato_vendas.to_sql("fato_vendas", conn, if_exists = "append", index = False)

conn.commit()

In [475]:
pd.read_sql_query("SELECT * FROM fato_vendas", conn)

,id_venda,id_cliente,id_produto,id_pagamento,id_data,preco
0,1,1,1,1,1,1000.0
1,2,2,1,2,1,500.0
2,3,3,2,3,1,1000.0
3,4,4,1,2,1,500.0
4,5,5,2,3,1,1000.0
...,...,...,...,...,...,...
7004,7005,6026,1,7,1095,500.0
7005,7006,6027,3,4,1095,2000.0
7006,7007,1582,3,1,1095,2000.0
7007,7008,5765,1,4,1095,500.0


In [476]:
cur.execute("""
INSERT OR IGNORE INTO dim_cliente(id_cliente,cliente, email, ddd, telefone)
VALUES(?,?,?,?, ?)""", (158000, "Julia", "julia.braz@xpweiun.com","11", "973438435"))

conn.commit()

Códio para atualização de registros no banco

In [477]:
df_novo = pd.read_excel("nova_base_vendas.xlsx")

In [478]:
df_novo = df_novo.rename(columns={
    "Produto" : "produto",
    "Preço" : "preco",
    "Pgto." : "forma_pagamento",
    "Nº Parcelas" : "num_parcelas",
    "Data" : "data",
    "Cliente" : "cliente",
    "E-mail" : "email",
    "DDD" : "ddd",
    "Telefone" : "telefone"
})

In [479]:
db_name = "dw_vendas.db"
conn = sqlite3.connect(db_name)
cur = conn.cursor()

In [480]:
dim_cliente_db = pd.read_sql("SELECT * FROM dim_cliente", conn)
dim_produtos_db = pd.read_sql("SELECT * FROM dim_produto", conn)
dim_pagamentos_db = pd.read_sql("SELECT * FROM dim_pagamento", conn)
dim_calendario_db = pd.read_sql("SELECT * FROM dim_calendario", conn)
fato_vendas_db = pd.read_sql("SELECT * FROM fato_vendas", conn)

In [481]:
df_novo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   produto          3 non-null      object        
 1   preco            3 non-null      int64         
 2   forma_pagamento  3 non-null      object        
 3   num_parcelas     3 non-null      int64         
 4   data             3 non-null      datetime64[ns]
 5   cliente          3 non-null      object        
 6   email            3 non-null      object        
 7   ddd              3 non-null      int64         
 8   telefone         3 non-null      int64         
dtypes: datetime64[ns](1), int64(4), object(4)
memory usage: 348.0+ bytes


In [482]:
df_novo["ddd"] = df_novo["ddd"].astype(str)
df_novo["telefone"] = df_novo["telefone"].astype(str)

In [483]:
clientes_novos = df_novo[["cliente", "email", "ddd", "telefone"]].drop_duplicates()

In [484]:
clientes_merged = clientes_novos.merge(
    dim_cliente_db,
    on=["cliente", "email", "ddd", "telefone"],
    how="left",
    indicator = True
)

In [485]:
clientes_merged

,cliente,email,ddd,telefone,id_cliente,_merge
0,Jonatas Martins,jonatas@gmail.com,92,984300000,1.0,both
1,Jhonns Martins,jhonns@hotmail.com,85,999000000,2.0,both
2,Iago Braz,iago@gmail.com,31,973438435,NaN,left_only


In [486]:
clientes_merged[clientes_merged["_merge"] == "left_only"]

,cliente,email,ddd,telefone,id_cliente,_merge
2,Iago Braz,iago@gmail.com,31,973438435,NaN,left_only


In [487]:
clientes_inserir_bd = clientes_merged[clientes_merged["_merge"] == "left_only"]

In [488]:
clientes_inserir_bd = clientes_inserir_bd[["cliente",	"email",	"ddd",	"telefone"]]

In [489]:
clientes_inserir_bd.to_sql("dim_cliente", conn, if_exists = "append", index = False)

1

In [490]:
produtos_novos = df_novo[["produto"]].drop_duplicates()

In [491]:
produtos_merged = produtos_novos.merge(
    dim_produtos_db,
    on=["produto"],
    how="left",
    indicator = True
)

In [492]:
produtos_inserir_bd = produtos_merged[produtos_merged["_merge"] == "left_only"][["produto"]]

In [493]:
produtos_inserir_bd.to_sql("dim_produto", conn, if_exists = "append", index = False)

1

In [494]:
pd.read_sql("SELECT * FROM dim_produto", conn)

,id_produto,produto
0,1,Produto A
1,2,Produto B
2,3,Produto C
3,4,Produto D


In [495]:
pg_novos = df_novo[["forma_pagamento", "num_parcelas"]].drop_duplicates()

In [496]:
pg_merged = pg_novos.merge(
    dim_pagamentos_db,
    on=["forma_pagamento", "num_parcelas"],
    how="left",
    indicator = True
)

In [497]:
pg_inserir_db = pg_merged[pg_merged["_merge"] == "left_only"][["forma_pagamento", "num_parcelas"]]

In [498]:
pg_inserir_db.to_sql("dim_pagamento", conn, if_exists = "append", index = False)

1

In [499]:
pd.read_sql("SELECT * FROM dim_pagamento", conn)

,id_pagamento,forma_pagamento,num_parcelas
0,1,Cartão de Credito,1
1,2,Cartão de Credito,2
2,3,Cartão de Credito,12
3,4,Boleto Bancário,1
4,5,Cartão de Credito,11
5,6,Cartão de Credito,6
6,7,Cartão de Credito,4
7,8,Cartão de Credito,3
8,9,Cartão de Credito,10
9,10,Cartão de Credito,5


In [500]:
dim_calendario_db["data"] = pd.to_datetime(dim_calendario_db["data"])

In [501]:
dim_calendario_db["data"]

,data
0,2018-01-01
1,2018-01-02
2,2018-01-03
3,2018-01-04
4,2018-01-05
...,...
1090,2020-12-27
1091,2020-12-28
1092,2020-12-29
1093,2020-12-30


In [502]:
tempo_novos = df_novo[["data"]].drop_duplicates()
tempo_novos["ano"] = tempo_novos["data"].dt.year
tempo_novos["mes"] = tempo_novos["data"].dt.month
tempo_novos["dia"] = tempo_novos["data"].dt.day

In [503]:
tempo_novos

,data,ano,mes,dia
0,2018-01-01,2018,1,1
2,2025-12-11,2025,12,11


In [504]:
tempo_merged = tempo_novos.merge(
    dim_calendario_db,
    on=["data", "ano", "mes", "dia"],
    how="left",
    indicator = True
)

In [505]:
tempo_inserir_db = tempo_merged[tempo_merged["_merge"] == "left_only"][["data", "ano", "mes", "dia"]]

In [506]:
tempo_inserir_db.to_sql("dim_calendario", conn, if_exists = "append", index = False)

1

In [507]:
pd.read_sql("SELECT * FROM dim_calendario", conn)

,id_data,data,dia,mes,ano
0,1,2018-01-01 00:00:00,1,1,2018
1,2,2018-01-02 00:00:00,2,1,2018
2,3,2018-01-03 00:00:00,3,1,2018
3,4,2018-01-04 00:00:00,4,1,2018
4,5,2018-01-05 00:00:00,5,1,2018
...,...,...,...,...,...
1091,1092,2020-12-28 00:00:00,28,12,2020
1092,1093,2020-12-29 00:00:00,29,12,2020
1093,1094,2020-12-30 00:00:00,30,12,2020
1094,1095,2020-12-31 00:00:00,31,12,2020


In [508]:
dim_clientes_db = pd.read_sql("SELECT * FROM dim_cliente", conn)
dim_produtos_db = pd.read_sql("SELECT * FROM dim_produto", conn)
dim_pagamentos_db = pd.read_sql("SELECT * FROM dim_pagamento", conn)
dim_calendario_db = pd.read_sql("SELECT * FROM dim_calendario", conn)
fato_vendas_db = pd.read_sql("SELECT * FROM fato_vendas", conn)

In [509]:
dim_calendario_db["data"] = pd.to_datetime(dim_calendario_db["data"])

In [510]:
fato_vendas_nova = df_novo.merge(
    dim_clientes_db,
    on= ["cliente", "email", "ddd", "telefone"],
    how= "left")

fato_vendas_nova = fato_vendas_nova.merge(
    dim_produtos_db,
    on = ["produto"],
    how = "left")

fato_vendas_nova = fato_vendas_nova.merge(
    dim_pagamentos_db,
    on = ["forma_pagamento", "num_parcelas"],
    how = "left")

fato_vendas_nova = fato_vendas_nova.merge(
    dim_calendario_db[["id_data", "data"]],
    on = 'data',
    how = "left"
    )


In [511]:
fato_vendas_nova = fato_vendas_nova[["id_cliente", "id_produto",	"id_pagamento",	"id_data", "preco"]]

In [512]:
fato_vendas_nova["preco"] =  fato_vendas_nova["preco"].astype(float)
fato_vendas_db["preco"] =  fato_vendas_db["preco"].astype(float)

In [513]:
fato_vendas_nova

,id_cliente,id_produto,id_pagamento,id_data,preco
0,1,1,1,1,1000.0
1,2,1,2,1,500.0
2,158001,4,24,1096,3000.0


In [514]:
fato_vendas_nova["chave"] = (
    fato_vendas_nova["id_cliente"].astype(str) + '|'+
    fato_vendas_nova["id_produto"].astype(str) + '|' +
    fato_vendas_nova["id_pagamento"].astype(str) + '|' +
    fato_vendas_nova["id_data"].astype(str) + '|' +
    fato_vendas_nova["preco"].astype(str)
)

fato_vendas_db["chave"] = (
    fato_vendas_db["id_cliente"].astype(str) + '|'+
    fato_vendas_db["id_produto"].astype(str) + '|' +
    fato_vendas_db["id_pagamento"].astype(str) + '|' +
    fato_vendas_db["id_data"].astype(str) + '|' +
    fato_vendas_db["preco"].astype(str)
)

In [515]:
chaves_existentes = set(fato_vendas_db["chave"])

df_fato_nova_final = fato_vendas_nova[~fato_vendas_nova["chave"].isin(chaves_existentes)]

In [516]:
df_fato_nova_final.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1 entries, 2 to 2
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_cliente    1 non-null      int64  
 1   id_produto    1 non-null      int64  
 2   id_pagamento  1 non-null      int64  
 3   id_data       1 non-null      int64  
 4   preco         1 non-null      float64
 5   chave         1 non-null      object 
dtypes: float64(1), int64(4), object(1)
memory usage: 56.0+ bytes


In [517]:
df_fato_nova_final = df_fato_nova_final[["id_cliente",	"id_produto",	"id_pagamento",	"id_data", "preco"]]

df_fato_nova_final.to_sql("fato_vendas", conn, if_exists= "append", index = False)

1

In [520]:
pd.read_sql("SELECT * FROM fato_vendas", conn)

,id_venda,id_cliente,id_produto,id_pagamento,id_data,preco
0,1,1,1,1,1,1000.0
1,2,2,1,2,1,500.0
2,3,3,2,3,1,1000.0
3,4,4,1,2,1,500.0
4,5,5,2,3,1,1000.0
...,...,...,...,...,...,...
7005,7006,6027,3,4,1095,2000.0
7006,7007,1582,3,1,1095,2000.0
7007,7008,5765,1,4,1095,500.0
7008,7009,5987,1,3,1095,500.0
